Project Settings to change:
1. Enable Identity-Aware Proxy
2. Enable OS Login (Not sure about this, currently in my test project it's disabled, but I heard best practice is to keep it enabled)
   1. https://cloud.google.com/compute/docs/oslogin/set-up-oslogin#gcloud
3. Please consider adding the IAP-secured Tunnel User IAM role (iap.tunnelInstances.accessViaIAP) to start using Cloud IAP for TCP forwarding for better performance.

Run the following in Cloud Shell in GCP to create the VM:

Change the following first:
instance name
metadata should be empty first
service account should be your service account

In [ ]:
gcloud compute instances create drg-data-pipeline \
    --project=drg-pipeline \
    --zone=us-central1-a \
    --machine-type=n2d-standard-8 \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=^,@^enable-osconfig=TRUE,@enable-oslogin=TRUE,@ssh-keys=resur:ssh-rsa\ \
AAAAB3NzaC1yc2EAAAADAQABAAABgQCvskXKEaH3gmIWUXYL7RKhA986gXZGYLfoOdUczxgPZ68HBBtpyOhifB22GprL9cYsI4QApr5lg3S5BRWjzYT1VSxPMW1dqcLvVbncfjwjYHH4FhY0fk7ijFihR2B6wQbE2LquZW3OtDcIfUP/zSE3IQ0Q9EaygJPeTItrLbEEwZ89kVAXX2knIG84xr5WClTxbP782xYoqg8tyQ25sr\+dJD9dsEGGgRnm\+d9J6D5a4UyiFjb5qU/0H5emv9ZX/20rufr5DEJAgOWw9umtwcTuwp/eRRh\+Id3y3BE3Q\+lQvd\+yZQXV3O5HPiC0Rw06\+dRs3gsROJaXqFG4dtsQ7SvMLvDbsXJwgFGyAyslKIJwCvOCnvWD7l0Tca5KydUkQttdDET08hkIj3yNLI\+P\+IdVCPVFFSACCgQ3HIH7nIPiVPRIP1q4d2f6M1PzuG87zaUOhjJBvYPz7kGhCNhEZL77b4dhVOS7LOTnv6AKbEKRjo9p91Wxj9IHc4nECcE4smE=\ resur$'\n'resurreccion_cmc:ecdsa-sha2-nistp256\ AAAAE2VjZHNhLXNoYTItbmlzdHAyNTYAAAAIbmlzdHAyNTYAAABBBE8qW7rtzzuBFcg6RrzAcmZHguh\+YCLhMWIJdngKnM4OF9O/fWHlYDYyyE0/SOYrbneLNjLWZ2wUR45KQFEj7tM=\ google-ssh\ \{\"userName\":\"resurreccion.cmc@gmail.com\",\"expireOn\":\"2024-08-06T15:54:20\+0000\"\}$'\n'resurreccion_cmc:ssh-rsa\ AAAAB3NzaC1yc2EAAAADAQABAAABAQCbu21YIOVt0hhjv81ZR4PQ7QxvMNzk/gqDwhZuHO\+Yh1y\+TJMk3pTr\+MkU7iiSVpfJPXMxzahRMxe6UxZjZQ93aJ8dDTThPYjLVdL9PpZ932xtfXYRWBinU\+yzMsEYZUyxo0d5NRpirK0rH\+7olyB0yiDucGc4NP2m3XYwEVhQRnkKdsS6X1F1wWMCJNaUBe4ONM5fcVqmowbFAw7Fv2TSfwDYYTatzSfcYxaP2yXZyItnQJ8AVi2g03zesTftboi66YfJFFVkS8dBh7b/GNaTzTU/gchHAdFYKIqNBQwfrCA9z5UP06FXQ\+NZRT5Zitz7pCrJ\+BdZSIcBUSfu9plb\ google-ssh\ \{\"userName\":\"resurreccion.cmc@gmail.com\",\"expireOn\":\"2024-08-06T15:54:27\+0000\"\} \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=271591364028-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/cloud-platform \
    --tags=http-server,https-server,lb-health-check \
    --create-disk=auto-delete=yes,boot=yes,device-name=drg-data-pipeline,image=projects/ubuntu-os-cloud/global/images/ubuntu-2404-noble-amd64-v20240726,mode=rw,size=100,type=pd-ssd \
    --shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any

Configure code tunnel to work on the VM

In [ ]:
sudo snap install code --classic
code tunnel
code tunnel service install
code tunnel
sudo loginctl enable-linger $USER
sudo loginctl enable-linger root
code tunnel service log
# open the link in the log and type in the authentication code XXXX-XXXX

Create ssh keys (run in windows terminal)

In [ ]:
# ssh-keygen -t rsa -f "C:\\Users\\resur\\.ssh\\gcp-resurreccion_cmc_gmail_com" -C resurreccion_cmc_gmail_com # change resurreccion_cmc_gmail_com to your username

Run this on windows terminal to get the HostName and HostKeyAlias

In [ ]:
# gcloud compute ssh drg-data-pipeline --dry-run --tunnel-through-iap

In [ ]:
# # Output will be something like this:

# C:\Users\resur>gcloud compute ssh drg-data-pipeline --dry-run --tunnel-through-iap

# No zone specified. Using zone [us-central1-a] for instance: [drg-data-pipeline].

# "C:\Users\resur\AppData\Local\Google\Cloud SDK\google-cloud-sdk\bin\sdk\putty.exe" -t -i "D:\Documents (D)\OneDrive - Philippine Institute for Development Studies\DRG\.ssh\google_compute_engine.ppk" -proxycmd ""C:\\Users\\resur\\AppData\\Local\\Google\\Cloud SDK\\google-cloud-sdk\\bin\\..\\platform\\bundledpython\\python.exe" "-S" "C:\\Users\\resur\\AppData\\Local\\Google\\Cloud SDK\\google-cloud-sdk\\bin\\..\\lib\\gcloud.py" compute start-iap-tunnel "drg-data-pipeline" "%port" --listen-on-stdin --project=drg-pipeline --zone=us-central1-a --verbosity=warning" resur@compute.43711418562236796

The below is the config file stored at C:\Users\username\\.ssh\config

In [ ]:
# Host drg-data-pipeline # CHANGE INSTANCE NAME
#     HostName compute.1853159480848533824 # CHANGE THE NUMBER AFTER THE PERIOD
#     User resurreccion_cmc_gmail_com # CHANGE THIS TO YOUR USERNAME
#     IdentityFile "C:\\Users\\resur\\.ssh\\gcp-resurreccion_cmc_gmail_com" # CHANGE USERNAME, GENERATE YOUR OWN IDENTITY FILE, UPLOAD PUBLIC KEY TO VM
#     CheckHostIP no
#     HashKnownHosts no
#     HostKeyAlias compute.1853159480848533824 # CHANGE THE NUMBER AFTER THE PERIOD
#     IdentitiesOnly yes
#     StrictHostKeyChecking yes # START WITH THIS DISABLED, THEN ENABLE AFTER FIRST RUN
#     UserKnownHostsFile "C:\\Users\\resur\\.ssh\\google_compute_known_hosts" # CHANGE USERNAME TO YOUR OWN
#     ProxyCommand "C:\\Users\\resur\\.pyenv\\pyenv-win\\versions\\3.12.4\\python.exe" "C:\\Users\\resur\\AppData\\Local\\Google\\Cloud SDK\\google-cloud-sdk\\bin\\..\\lib\\gcloud.py" compute start-iap-tunnel "test-drg-data-pipeline" "%p" --listen-on-stdin --project=test-drg-pipeline --zone=us-central1-a --verbosity=warning # CHANGE USERNAME, PYENV PYTHON VERSION, INSTANCE NAME, PROJ NAME, ZONE
#     ProxyUseFdpass no


Save the above as ~/.ssh/config

Connect with StrictHostKeyChecking no first then enable it later

Then ssh into the VM with the below command

In [ ]:
# ssh drg-data-pipeline

VM Setup commands:

In [ ]:
# Update the package list:
sudo apt update

# Install Jupyter:
sudo apt install jupyter jupyter-core jupyter-client build-essential libcurl4-openssl-dev libssl-dev libxml2-dev

# upgrade packages
sudo apt upgrade

VM R installation

In [ ]:
# Install R
## Update package list
sudo apt update
sudo apt install -y software-properties-common dirmngr
## Add CRAN GPG Key
wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc | sudo tee -a /etc/apt/trusted.gpg.d/cran_ubuntu_key.asc
## Add CRAN repository
sudo add-apt-repository 'deb https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/'
## Update package list again
sudo apt update
## Install R 4.4.1
sudo apt install -y r-base
## Check R Version
R --version

R package installation (sudo to install to site-wide library)

In [ ]:
sudo R

Package installation

In [ ]:
install.packages("languageserver")
install.packages("jsonlite")
install.packages("rlang")
install.packages("yaml")
install.packages("IRkernel")
install.packages("here")


Register the R kernel as a jupyter kernel system-wide

In [ ]:
IRkernel::installspec(user = FALSE)


In [ ]:
quit()

The following commented out code is for running jupyter lab, not VS code

In [ ]:
# R

In [ ]:
# install.packages("languageserver")
# install.packages("jsonlite")
# install.packages("rlang")
# install.packages("yaml")
# install.packages("IRkernel")
# install.packages("here")


In [ ]:
# jupyter notebook --generate-config
# jupyter notebook stop 
# jupyter notebook
# nano /home/resur/.jupyter/jupyter_notebook_config.py


In [ ]:
# c.NotebookApp.kernel_spec_manager_class = 'jupyter_client.kernelspec.KernelSpecManager'
# c.KernelSpecManager.ensure_native_kernel = False
# c.KernelSpecManager.whitelist = set(['python3', 'ir'])


In [ ]:
# jupyter notebook stop 
# jupyter notebook
# export JUPYTER_PATH=/usr/local/share/jupyter/kernels


In [ ]:
# sudo R

In [ ]:
# install.packages('IRkernel')
# IRkernel::installspec(user = FALSE)
# quit()


Check jupyter kernels

In [ ]:
jupyter kernelspec list


Make system-wide libraries writable by R

Otherwise, we'd need to rely on renv and I haven't gotten that to work

In [ ]:
sudo chmod -R 777 /usr/local/lib/R/site-library
sudo chmod -R 777 /usr/lib/R/site-library
sudo mkdir -P /home/data
sudo chmod -R 777 /home/data
sudo chown -R root:root /home/data


Install gcloud CLI on the VM

In [ ]:
sudo apt-get update

sudo apt-get install apt-transport-https ca-certificates gnupg curl

curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo gpg --dearmor -o /usr/share/keyrings/cloud.google.gpg

echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt cloud-sdk main" | sudo tee -a /etc/apt/sources.list.d/google-cloud-sdk.list

sudo apt-get update && sudo apt-get install google-cloud-cli

gcloud init

Configure git on the VM

In [ ]:
git config --global user.name "Carlos Miguel Resurreccion"
git config --global user.email resurreccion.cmc@gmail.com

Configure Patch

In [ ]:
# https://console.cloud.google.com/compute/patch/dashboard enable patch
# https://console.cloud.google.com/apis/api/osconfig.googleapis.com/overview
# Manually install OS Config agent
sudo su -c "echo 'deb http://packages.cloud.google.com/apt google-compute-engine-focal-stable main' > \
/etc/apt/sources.list.d/google-compute-engine.list"

sudo apt update
sudo apt -y install google-osconfig-agent

# Apply OS Config to all instances in a project
gcloud compute project-info add-metadata --project drg-pipeline --metadata=enable-osconfig=TRUE

# Apply OS Config to an individual VM
gcloud compute instances add-metadata drg-data-pipeline --metadata=enable-osconfig=TRUE

# Apply OS Config when creating an instance
gcloud compute instances create drg-data-pipeline --metadata=enable-osconfig=TRUE

# Enable full VM Manager features
gcloud compute os-config project-feature-settings update --project drg-pipeline --patch-and-config-feature-set=full

# Verify it's working
gcloud compute os-config project-feature-settings describe --project drg-pipeline

Link /home/data to your username's drg-pipeline/data-cleaning/data folder

In [ ]:
sudo ln -s /home/data /home/resur/drg-pipeline/data-cleaning/data # change resur to your username